<a href="https://colab.research.google.com/github/tsukki8/ds2002-fa26/blob/main/notebooks/06-json/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [2]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [5]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(
    orders_json,
    record_path="lines",
    meta=["order", "vendor_id"]
)

assert len(lines) == 4
assert lines["qty"].sum() == 7

print(lines)

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [6]:
# TODO
joined = lines.merge(
    pd.DataFrame(vendors_json),
    on="vendor_id",
    how="left",
    indicator=True
)

print(joined)

unmatched = joined[joined["_merge"] == "left_only"]

print("\nUnmatched vendor:")
print(unmatched)

print("\nUnits attached to unmatched vendor:",
      unmatched["qty"].sum())

           item  qty order vendor_id          name zone _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A   both
1          Soda    1     1      V-01  Hoos Burgers    A   both
2   Foam Finger    1     2      V-10     Cav Merch    A   both
3       Hot Dog    3     3      V-01  Hoos Burgers    A   both

Unmatched vendor:
Empty DataFrame
Columns: [item, qty, order, vendor_id, name, zone, _merge]
Index: []

Units attached to unmatched vendor: 0


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [7]:
# TODO
joined["vendor_name"] = joined["name"].fillna("Unknown vendor")

vendor_totals = (
    joined
    .groupby("vendor_name")["qty"]
    .sum()
    .sort_values(ascending=False)
)

print(vendor_totals)

vendor_name
Hoos Burgers    6
Cav Merch       1
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [8]:
# TODO
zone_totals = (
    joined
    .groupby("zone")["qty"]
    .sum()
    .sort_values(ascending=False)
)

print(zone_totals)

print("\nTotal units represented in zones:",
      zone_totals.sum())

print("Total units overall:",
      joined["qty"].sum())

print("Units without a zone:",
      joined.loc[joined["zone"].isna(), "qty"].sum())

zone
A    7
Name: qty, dtype: int64

Total units represented in zones: 7
Total units overall: 7
Units without a zone: 0


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [9]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged_lines = []

for order in ragged_json:
    for line in order.get("lines", []):
        ragged_lines.append({
            "order": order["order"],
            "vendor_id": order["vendor_id"],
            "item": line.get("item"),
            "qty": line.get("qty")
        })

ragged_lines = pd.DataFrame(ragged_lines)

print(ragged_lines)
print("\nDtypes:")
print(ragged_lines.dtypes)

   order vendor_id   item  qty
0      4      V-01   Soda  2.0
1      6      V-01  Fries  NaN

Dtypes:
order          int64
vendor_id     object
item          object
qty          float64
dtype: object


### Q6 — Validate

In [10]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) The 'one row per line item' is a useful shape here because each item and its quantity value are analyzed separately. It is easy to perform operations on these items if they are one row per order and its unnecessary to spend time aggregating nested items in a line.
b) A missing quantity is when it's not provided or unknown, and a quantity of zero means that no units were sold for certain. Treating a missing quantity as zero could make the data look more complete than reality and result in incorrect totals/averages.